# Group DRO — Expériences sur Waterbirds

Basé sur : *Distributionally Robust Neural Networks for Group Shifts* — Sagawa et al., 2021

**Plan du notebook :**
1. Setup GPU + dépendances
2. Téléchargement dataset Waterbirds
3. Baseline ERM
4. Group DRO
5. Ablations (weight_decay, reweighting)
6. Visualisation et analyse

> **Runtime** : `Runtime > Change runtime type > T4 GPU`

## 1. Vérification GPU & montage Drive

In [ ]:
import torch
print('GPU disponible :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('⚠️  Pas de GPU — aller dans Runtime > Change runtime type > T4 GPU')

In [ ]:
# Monter Google Drive pour persister les résultats (optionnel mais recommandé)
USE_DRIVE = True  # mettre False si vous ne voulez pas utiliser Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/group_DRO_experiments'
else:
    BASE_DIR = '/content/group_DRO_experiments'

import os
os.makedirs(BASE_DIR, exist_ok=True)
print('Répertoire de travail :', BASE_DIR)

## 2. Installation des dépendances et clonage du repo

In [ ]:
%%bash
pip install pytorch_transformers wilds -q
# pytorch_transformers est requis même pour ResNet (import en top-level dans train.py)

In [ ]:
import os

REPO_DIR = '/content/group_DRO'

if not os.path.exists(REPO_DIR):
    os.system('git clone https://github.com/kohpangwei/group_DRO.git /content/group_DRO')
    print('Repo cloné.')
else:
    print('Repo déjà présent.')

os.chdir(REPO_DIR)
print('Working dir :', os.getcwd())

In [ ]:
# ── Patches de compatibilité Colab ──────────────────────────────────────────
#
# 1. train.py importe pytorch_transformers en top-level même pour ResNet.
#    On remplace par transformers (déjà installé dans Colab) avec un fallback.
#
# 2. run_expt.py fixe num_workers=4 ce qui peut bloquer le fork dans Colab.
#    On le réduit à 2.

import re

# ── Patch 1 : train.py ──
train_path = '/content/group_DRO/train.py'
with open(train_path) as f:
    src = f.read()

old_import = 'from pytorch_transformers import AdamW, WarmupLinearSchedule'
new_import = (
    'try:\n'
    '    from pytorch_transformers import AdamW, WarmupLinearSchedule\n'
    'except ImportError:\n'
    '    from transformers import AdamW\n'
    '    from transformers import get_linear_schedule_with_warmup as WarmupLinearSchedule\n'
)
if old_import in src:
    src = src.replace(old_import, new_import)
    with open(train_path, 'w') as f:
        f.write(src)
    print('✅ train.py patché (pytorch_transformers → transformers fallback)')
else:
    print('ℹ️  train.py déjà patché ou import différent')

# ── Patch 2 : run_expt.py num_workers ──
expt_path = '/content/group_DRO/run_expt.py'
with open(expt_path) as f:
    src = f.read()

if "'num_workers':4" in src:
    src = src.replace("'num_workers':4", "'num_workers':2")
    with open(expt_path, 'w') as f:
        f.write(src)
    print('✅ run_expt.py patché (num_workers 4 → 2)')
else:
    print('ℹ️  run_expt.py déjà patché ou num_workers différent')

# ── Vérification rapide : import sans erreur ──
import subprocess, sys
check = subprocess.run(
    [sys.executable, '-c', 'import sys; sys.path.insert(0,"/content/group_DRO"); import train'],
    capture_output=True, text=True
)
if check.returncode == 0:
    print('✅ import train OK')
else:
    print('❌ import train ERREUR :')
    print(check.stderr[-2000:])

## 3. Téléchargement du dataset Waterbirds

In [ ]:
import os

DATA_DIR = os.path.join(BASE_DIR, 'data')
WATERBIRDS_DIR = os.path.join(DATA_DIR, 'waterbird_complete95_forest2water2')
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(WATERBIRDS_DIR):
    print('Téléchargement de Waterbirds (~1.2 GB)...')
    url = 'https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz'
    os.system(f'wget -q --show-progress -O {DATA_DIR}/waterbirds.tar.gz "{url}"')
    os.system(f'tar -xzf {DATA_DIR}/waterbirds.tar.gz -C {DATA_DIR}/')
    os.system(f'rm {DATA_DIR}/waterbirds.tar.gz')
    print('Dataset extrait dans :', WATERBIRDS_DIR)
else:
    print('Dataset déjà présent :', WATERBIRDS_DIR)

# Vérification
assert os.path.exists(WATERBIRDS_DIR), f'Dataset non trouvé : {WATERBIRDS_DIR}'
files = os.listdir(WATERBIRDS_DIR)
print(f'{len(files)} éléments dans le dataset')

## 4. Fonction utilitaire pour lancer les expériences

In [ ]:
import subprocess, os, sys

REPO_DIR = '/content/group_DRO'
LOGS_DIR = os.path.join(BASE_DIR, 'logs')
os.makedirs(LOGS_DIR, exist_ok=True)

def run_experiment(name, extra_args='', n_epochs=50, weight_decay=1e-4,
                   robust=False, reweight_groups=False,
                   gamma=0.1, generalization_adjustment='0',
                   lr=0.001, batch_size=128):
    """
    Lance une expérience group_DRO sur Waterbirds (CUB dataset).

    Args:
        name: identifiant du run (dossier de logs)
        robust: True = Group DRO, False = ERM
        reweight_groups: True = reweighting par taille de groupe
        weight_decay: L2 régularisation
        gamma: pas EMA de la loss par groupe (DRO seulement)
        generalization_adjustment: ajustement de généralisation
    """
    log_dir = os.path.join(LOGS_DIR, name)
    os.makedirs(log_dir, exist_ok=True)

    cmd = [
        sys.executable, f'{REPO_DIR}/run_expt.py',
        '-s', 'confounder',
        '-d', 'CUB',
        '-t', 'waterbird_complete95',
        '-c', 'forest2water2',
        '--root_dir', WATERBIRDS_DIR,
        '--model', 'resnet50',
        '--n_epochs', str(n_epochs),
        '--batch_size', str(batch_size),
        '--lr', str(lr),
        '--weight_decay', str(weight_decay),
        '--gamma', str(gamma),
        '--generalization_adjustment', str(generalization_adjustment),
        '--log_dir', log_dir,
        '--save_best',
        '--save_last',
    ]

    if robust:
        cmd.append('--robust')
    if reweight_groups:
        cmd.append('--reweight_groups')
    if extra_args:
        cmd.extend(extra_args.split())

    print(f'\n=== Expérience : {name} ===')
    print(' '.join(cmd[2:]))
    print()

    # Stream stdout en temps réel, capturer stderr pour affichage en cas d'erreur
    process = subprocess.Popen(
        cmd,
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    stderr_output = process.stderr.read()
    process.wait()

    if process.returncode != 0:
        print(f'\n❌ Erreur (code {process.returncode})')
        if stderr_output:
            print('\n--- STDERR (50 dernières lignes) ---')
            lines = stderr_output.strip().split('\n')
            print('\n'.join(lines[-50:]))
    else:
        print(f'\n✅ Terminé — logs dans {log_dir}')

    return log_dir

print('Fonction run_experiment() définie.')

## 5. Expérience 1 — Baseline ERM (Empirical Risk Minimization)

ERM = minimisation de la perte moyenne, sans tenir compte des groupes.

**Attendu** : bonne accuracy moyenne, mais mauvaise worst-group accuracy (problème des spurious correlations).

In [ ]:
erm_dir = run_experiment(
    name='erm_baseline',
    robust=False,
    reweight_groups=False,
    n_epochs=100,
    weight_decay=1e-4,
    lr=0.001,
    batch_size=128
)

## 6. Expérience 2 — Group DRO

Group DRO = minimisation de la perte du **pire sous-groupe** via des probabilités adversariales.

**Attendu** : worst-group accuracy significativement meilleure qu'ERM.

In [ ]:
dro_dir = run_experiment(
    name='group_dro',
    robust=True,
    reweight_groups=False,
    n_epochs=100,
    weight_decay=1e-4,
    gamma=0.1,
    generalization_adjustment='0',
    lr=0.001,
    batch_size=128
)

## 7. Expérience 3 — Ablation : impact de la régularisation (weight_decay)

**Hypothèse du papier** : la régularisation L2 est cruciale pour que Group DRO généralise bien sur le pire groupe.

In [ ]:
# Ablation weight_decay avec Group DRO
wd_values = [1e-5, 1e-4, 1e-3, 1e-2]
wd_dirs = {}

for wd in wd_values:
    wd_str = f'{wd:.0e}'.replace('-0', '-')
    name = f'dro_wd_{wd_str}'
    wd_dirs[wd] = run_experiment(
        name=name,
        robust=True,
        n_epochs=100,
        weight_decay=wd,
        gamma=0.1,
        lr=0.001,
        batch_size=128
    )

## 8. Expérience 4 — Reweighting vs DRO vs ERM

Comparaison de 3 stratégies :
- **ERM** : aucune correction
- **Reweighting** : pondère les samples par l'inverse de la taille du groupe
- **Group DRO** : optimise le pire groupe de façon adversariale

In [ ]:
reweight_dir = run_experiment(
    name='reweighting_only',
    robust=False,
    reweight_groups=True,
    n_epochs=100,
    weight_decay=1e-4,
    lr=0.001,
    batch_size=128
)

## 9. Analyse et visualisation des résultats

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Groupes Waterbirds :
# 0 = Waterbird sur fond eau (majoritaire, facile)
# 1 = Waterbird sur fond forêt (minoritaire, difficile)
# 2 = Landbird sur fond eau (minoritaire, difficile)
# 3 = Landbird sur fond forêt (majoritaire, facile)
GROUP_NAMES = [
    'Waterbird / Water bg',
    'Waterbird / Land bg',
    'Landbird / Water bg',
    'Landbird / Land bg'
]

def load_results(log_dir, split='val'):
    """Charge les métriques depuis les CSVs de logs."""
    csv_path = os.path.join(log_dir, f'{split}.csv')
    if not os.path.exists(csv_path):
        print(f'⚠️  {csv_path} non trouvé')
        return None
    df = pd.read_csv(csv_path)
    # Garder la dernière entrée par epoch
    df = df.groupby('epoch').last().reset_index()
    return df

def get_worst_group_acc(df):
    """Retourne l'accuracy du pire groupe par epoch."""
    group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
    return df[group_cols].min(axis=1)

def get_best_epoch(df, metric='worst_group'):
    """Retourne les stats à l'epoch avec la meilleure worst-group accuracy."""
    group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
    df = df.copy()
    df['worst_group_acc'] = df[group_cols].min(axis=1)
    best_idx = df['worst_group_acc'].idxmax()
    return df.loc[best_idx]

print('Fonctions d\'analyse définies.')

In [ ]:
# ─── Tableau récapitulatif ERM vs Group DRO ───

experiments = {
    'ERM': erm_dir,
    'Reweighting': reweight_dir,
    'Group DRO': dro_dir,
}

results_table = []

for exp_name, log_dir in experiments.items():
    df_val = load_results(log_dir, split='val')
    df_test = load_results(log_dir, split='test')
    if df_val is None:
        continue

    best = get_best_epoch(df_val)
    group_cols = [c for c in df_val.columns if c.startswith('avg_acc_group:')]

    row = {
        'Méthode': exp_name,
        'Avg Acc (val)': f"{best['avg_acc']:.3f}",
        'Worst-group Acc (val)': f"{best[group_cols].min():.3f}",
        'Epoch': int(best['epoch']),
    }
    for i, col in enumerate(group_cols):
        row[f'G{i} ({GROUP_NAMES[i].split("/")[0].strip()})'] = f"{best[col]:.3f}"

    results_table.append(row)

df_results = pd.DataFrame(results_table)
print('=== Résultats à la meilleure worst-group epoch (val) ===\n')
print(df_results.to_string(index=False))

In [ ]:
# ─── Courbes d'apprentissage : Avg Acc vs Worst-group Acc ───

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {'ERM': 'tab:blue', 'Reweighting': 'tab:orange', 'Group DRO': 'tab:green'}

for exp_name, log_dir in experiments.items():
    df = load_results(log_dir, split='val')
    if df is None:
        continue
    group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
    worst = df[group_cols].min(axis=1)
    c = colors.get(exp_name, 'gray')

    axes[0].plot(df['epoch'], df['avg_acc'], label=exp_name, color=c)
    axes[1].plot(df['epoch'], worst, label=exp_name, color=c)

for ax, title in zip(axes, ['Average Accuracy', 'Worst-Group Accuracy']):
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.suptitle('ERM vs Reweighting vs Group DRO — Waterbirds (val)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'curves_main.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Courbes par groupe : ERM vs DRO ───

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
group_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for ax, (exp_name, log_dir) in zip(axes, [('ERM', erm_dir), ('Group DRO', dro_dir)]):
    df = load_results(log_dir, split='val')
    if df is None:
        continue
    group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
    for i, col in enumerate(group_cols):
        ax.plot(df['epoch'], df[col], label=GROUP_NAMES[i], color=group_colors[i])
    ax.plot(df['epoch'], df['avg_acc'], label='Average', color='black', linestyle='--', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{exp_name} — Accuracy par groupe')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.suptitle('Accuracy par groupe (val) — Waterbirds', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'curves_per_group.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Visualisation training dynamics (train.csv) ───

def load_train_csv(log_dir):
    path = os.path.join(log_dir, 'train.csv')
    if not os.path.exists(path):
        print(f'⚠️  {path} non trouvé — expérience pas encore lancée ?')
        return None
    df = pd.read_csv(path)
    return df.groupby('epoch').last().reset_index()

train_dro_ep = load_train_csv(dro_dir)
train_erm_ep = load_train_csv(erm_dir)

if train_dro_ep is None or train_erm_ep is None:
    print('Cellule ignorée : lancer d\'abord les expériences ERM et Group DRO.')
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    group_cols = [c for c in train_dro_ep.columns if c.startswith('avg_acc_group:')]
    group_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

    for (title, df_ep, ax_loss, ax_acc) in [
        ('DRO',  train_dro_ep, axes[0, 0], axes[0, 1]),
        ('ERM',  train_erm_ep, axes[1, 0], axes[1, 1]),
    ]:
        for i, col in enumerate(group_cols):
            loss_col = col.replace('avg_acc', 'avg_loss')
            label = GROUP_NAMES[i] if i < len(GROUP_NAMES) else f'Group {i}'
            if loss_col in df_ep.columns:
                ax_loss.plot(df_ep['epoch'], df_ep[loss_col], label=label, color=group_colors[i % len(group_colors)])
            ax_acc.plot(df_ep['epoch'], df_ep[col], label=label, color=group_colors[i % len(group_colors)])

        ax_loss.set_title(f'{title} — Loss par groupe (train)')
        ax_loss.set_xlabel('Epoch'); ax_loss.legend(fontsize=8); ax_loss.grid(True, alpha=0.3)
        ax_acc.set_title(f'{title} — Accuracy par groupe (train)')
        ax_acc.set_xlabel('Epoch'); ax_acc.legend(fontsize=8); ax_acc.grid(True, alpha=0.3)

    plt.suptitle('Training dynamics — ERM vs Group DRO', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'training_dynamics.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ─── Visualisation adv_probs (DRO) : poids des groupes au fil du training ───
# (nécessite d'ajouter du logging dans loss.py — cellule optionnelle)

# Lire le fichier train.csv du run DRO
train_dro = pd.read_csv(os.path.join(dro_dir, 'train.csv'))
train_erm = pd.read_csv(os.path.join(erm_dir, 'train.csv'))

# Dernière entrée par epoch (fin de chaque epoch)
train_dro_ep = train_dro.groupby('epoch').last().reset_index()
train_erm_ep = train_erm.groupby('epoch').last().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

group_cols = [c for c in train_dro.columns if c.startswith('avg_acc_group:')]

# Loss par groupe (train)
ax = axes[0, 0]
for i, col in enumerate(group_cols):
    loss_col = col.replace('avg_acc', 'avg_loss')
    if loss_col in train_dro_ep.columns:
        ax.plot(train_dro_ep['epoch'], train_dro_ep[loss_col], label=GROUP_NAMES[i], color=group_colors[i])
ax.set_title('DRO — Loss par groupe (train)')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Acc par groupe (train) — DRO
ax = axes[0, 1]
for i, col in enumerate(group_cols):
    if col in train_dro_ep.columns:
        ax.plot(train_dro_ep['epoch'], train_dro_ep[col], label=GROUP_NAMES[i], color=group_colors[i])
ax.set_title('DRO — Accuracy par groupe (train)')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Loss par groupe (train) — ERM
ax = axes[1, 0]
for i, col in enumerate(group_cols):
    loss_col = col.replace('avg_acc', 'avg_loss')
    if loss_col in train_erm_ep.columns:
        ax.plot(train_erm_ep['epoch'], train_erm_ep[loss_col], label=GROUP_NAMES[i], color=group_colors[i])
ax.set_title('ERM — Loss par groupe (train)')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Acc par groupe (train) — ERM
ax = axes[1, 1]
for i, col in enumerate(group_cols):
    if col in train_erm_ep.columns:
        ax.plot(train_erm_ep['epoch'], train_erm_ep[col], label=GROUP_NAMES[i], color=group_colors[i])
ax.set_title('ERM — Accuracy par groupe (train)')
ax.set_xlabel('Epoch')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Training dynamics — ERM vs Group DRO', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'training_dynamics.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Tableau final récapitulatif

Reproduit (approximativement) le **Tableau 1** du papier.

In [ ]:
import pandas as pd

all_experiments = {
    'ERM': erm_dir,
    'Reweighting': reweight_dir,
    'Group DRO': dro_dir,
}
for wd, d in wd_dirs.items():
    all_experiments[f'DRO wd={wd:.0e}'] = d

rows = []
for exp_name, log_dir in all_experiments.items():
    for split in ['val', 'test']:
        df = load_results(log_dir, split=split)
        if df is None:
            continue
        best = get_best_epoch(df)
        group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
        rows.append({
            'Expérience': exp_name,
            'Split': split,
            'Avg Acc': round(best['avg_acc'], 3),
            'Worst Acc': round(best[group_cols].min(), 3),
            'Gap': round(best['avg_acc'] - best[group_cols].min(), 3),
            **{f'G{i}': round(best[col], 3) for i, col in enumerate(group_cols)},
            'Best Epoch': int(best['epoch'])
        })

df_final = pd.DataFrame(rows)
print('=== Tableau final : toutes expériences ===\n')
print(df_final.to_string(index=False))

# Sauvegarder
df_final.to_csv(os.path.join(BASE_DIR, 'results_summary.csv'), index=False)
print(f'\nSauvegardé dans {BASE_DIR}/results_summary.csv')

## 11. Analyse : tension Avg Acc vs Worst-group Acc

In [ ]:
# Scatter plot : avg accuracy vs worst-group accuracy pour tous les runs

fig, ax = plt.subplots(figsize=(9, 7))

plot_styles = {
    'ERM':         {'color': 'tab:blue',   'marker': 'o', 'size': 150},
    'Reweighting': {'color': 'tab:orange', 'marker': 's', 'size': 150},
    'Group DRO':   {'color': 'tab:green',  'marker': '^', 'size': 150},
}

for exp_name, log_dir in all_experiments.items():
    df = load_results(log_dir, split='val')
    if df is None:
        continue
    best = get_best_epoch(df)
    group_cols = [c for c in df.columns if c.startswith('avg_acc_group:')]
    avg = best['avg_acc']
    worst = best[group_cols].min()

    style = plot_styles.get(exp_name, {'color': 'gray', 'marker': 'D', 'size': 80})
    ax.scatter(avg, worst, label=exp_name,
               color=style['color'], marker=style['marker'], s=style['size'], zorder=5)
    ax.annotate(exp_name, (avg, worst), textcoords='offset points', xytext=(8, 4), fontsize=9)

# Ligne de référence : pas de gap
lim = [0.5, 1.0]
ax.plot(lim, lim, 'k--', alpha=0.3, label='Avg = Worst (idéal)')

ax.set_xlabel('Average Accuracy (val)')
ax.set_ylabel('Worst-Group Accuracy (val)')
ax.set_title('Trade-off : Accuracy Moyenne vs Pire Groupe\n(Waterbirds, meilleure epoch val)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.0, 1.0)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'tradeoff_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Résumé des résultats attendus

| Méthode | Avg Acc | Worst-Group Acc |
|---------|---------|----------------|
| ERM | ~97% | ~72% |
| Reweighting | ~93% | ~80% |
| Group DRO | ~92% | ~87% |

**Points clés :**
- ERM sacrifie les groupes minoritaires (Waterbird/Land bg, Landbird/Water bg)
- Group DRO améliore la worst-group acc **au prix** d'une légère baisse de l'accuracy moyenne
- La régularisation L2 est cruciale : sans elle, le modèle DRO sur-apprend sur le pire groupe en validation → mauvaise généralisation en test

**Paramètres du papier (Table 1, Waterbirds) :**
- lr = 1e-3, weight_decay = 1e-4, gamma = 0.1, 300 epochs, batch_size = 128